In [1]:
import sys
print(sys.executable)

C:\Users\chami\sepsis-early-warning\.venv\Scripts\python.exe


In [2]:
import sys
sys.path.append("..")
import duckdb
import pandas as pd

con = duckdb.connect("../data/sepsis.duckdb", read_only=True)

def q(sql):
    """Run a SQL query and return a pandas DataFrame."""
    return con.sql(sql).df()

In [3]:
q("""
SELECT hospital,
       COUNT(*)                                       AS all_patients,
       SUM((NOT included)::INT)                       AS excluded,
       SUM(included::INT)                             AS included_patients,
       SUM((included AND septic = 1)::INT)            AS included_septic,
       ROUND(AVG(septic) FILTER (WHERE included), 4)  AS septic_frac_after_exclusion
FROM cohort
GROUP BY hospital
ORDER BY hospital
""")

,hospital,all_patients,excluded,included_patients,included_septic,septic_frac_after_exclusion
0,A,20336,203.0,20133.0,1587.0,0.0788
1,B,20000,223.0,19777.0,919.0,0.0465


In [4]:
q("""
SELECT split,
       COUNT(*)              AS patients,
       SUM(septic)           AS septic_patients,
       ROUND(AVG(septic), 4) AS septic_patient_frac
FROM cohort
WHERE included
GROUP BY split
ORDER BY split
""")

,split,patients,septic_patients,septic_patient_frac
0,external,19777,919.0,0.0465
1,test,3021,239.0,0.0791
2,train,14092,1110.0,0.0788
3,val,3020,238.0,0.0788


In [5]:
q("""
SELECT split,
       COUNT(*)                   AS hours,
       ROUND(AVG(SepsisLabel), 4) AS positive_hour_frac
FROM cohort_observations
GROUP BY split
ORDER BY split
""")

,split,hours,positive_hour_frac
0,external,760117,0.0117
1,test,118406,0.0196
2,train,550591,0.0195
3,val,119450,0.0193


In [6]:
q("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT patient_id) AS patients,
       SUM((included AND split IS NULL)::INT) AS included_without_split
FROM cohort
""")

,rows,patients,included_without_split
0,40336,40336,0.0


## Findings (Part F: cohort and splits)

**Attrition**
| | A | B | Total |
|---|---|---|---|
| All patients | 20,336 | 20,000 | 40,336 |
| Excluded (septic from first recorded hour) | 203 | 223 | 426 |
| Included | 20,133 | 19,777 | 39,910 |
| Included septic | 1,587 (7.88%) | 919 (4.65%) | 2,506 |

**Splits**
| Split | Patients | Septic (rate) | Hours | Positive hours |
|---|---|---|---|---|
| Train (A) | 14,092 | 1,110 (7.88%) | 550,591 | 1.95% |
| Validation (A) | 3,020 | 238 (7.88%) | 119,450 | 1.93% |
| Test (A) | 3,021 | 239 (7.91%) | 118,406 | 1.96% |
| External (B) | 19,777 | 919 (4.65%) | 760,117 | 1.17% |

- Stratification worked: sepsis rates nearly identical across A's splits, at both patient and hour level
- Integrity: one row per patient, no included patient without a split
- Splits reproducible (identical on rerun)
- Excluded patients had only positive hours, so the positive-hour rate fell (A: 2.17% → ~1.95%); ~1 positive hour in 50 for training
- Prevalence gap between hospitals widened after exclusion (7.9% vs 4.6%): expect lower PPV and possible calibration shift at B